In [1]:
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np


df = pd.read_csv("/home/pavloops/PycharmProjects/pytorch_lightning_practice/final_project/data/dataset.csv")
product_info = pd.read_csv("/home/pavloops/PycharmProjects/pytorch_lightning_practice/final_project/data/product_info.csv")

print(df.columns)
df.shape

Index(['user_id', 'year_month', 'item_id', 'cnt', 'avg_price', 'brand',
       'category', 'country', 'inn', 'owner'],
      dtype='object')


(4720884, 10)

In [2]:
print(df.nunique().to_markdown())

|            |      0 |
|:-----------|-------:|
| user_id    |  29999 |
| year_month |     12 |
| item_id    |  11002 |
| cnt        |   1484 |
| avg_price  | 180930 |
| brand      |   4840 |
| category   |     16 |
| country    |     39 |
| inn        |   2005 |
| owner      |    580 |


In [3]:
print(df.isna().sum())

user_id            0
year_month         0
item_id            0
cnt                0
avg_price          0
brand              0
category      725969
country       700764
inn           820364
owner         700764
dtype: int64


## Negative Samples

In [4]:
df_pos = df.copy()
df_pos = df_pos[df_pos['brand'] != '_ПРОЧИЙ АССОРТИМЕНТ']
df_items = product_info.copy()
df_pos = df_pos.fillna('unknown')

In [5]:
TOP_N = 1000

df_items_top = (
    df_items
    .sort_values(['year_month', 'popularity'], ascending=False)
    .groupby('year_month')
    .head(TOP_N)
)

In [6]:
month2items = (
    df_items_top
    .groupby('year_month')['item_id']
    .apply(list)
    .to_dict()
)

In [7]:
user_month_pos = (
    df_pos
    .groupby(['user_id', 'year_month'])['item_id']
    .apply(set)
)

In [8]:
NEG_PER_POS = 3  # 1:3 — хороший баланс

rows = []

for (user, month), pos_items in tqdm(user_month_pos.items(), total=len(user_month_pos)):

    # кандидаты (популярные товары месяца)
    candidates = set(month2items.get(month, []))

    if not candidates:
        continue

    # убираем купленные
    neg_pool = list(candidates - pos_items)

    if len(neg_pool) == 0:
        continue

    # сколько негативов брать
    n_neg = min(len(pos_items) * NEG_PER_POS, len(neg_pool))

    neg_sample = np.random.choice(
        neg_pool,
        size=n_neg,
        replace=False
    )

    # --- позитивы ---
    for item in pos_items:
        rows.append((user, month, item, 1))

    # --- негативы ---
    for item in neg_sample:
        rows.append((user, month, item, 0))

df_train = pd.DataFrame(
    rows,
    columns=['user_id', 'year_month', 'item_id', 'label']
)

  0%|          | 0/342437 [00:00<?, ?it/s]

In [9]:
df_train.sample(10, random_state=777)

,user_id,year_month,item_id,label
2906591,2662904,202510,2227,0
107216,16468,202512,78991,0
4269718,4936527,202508,89207,0
5600198,5406757,202502,54916,0
6631685,5743875,202510,84727,1
15584689,8098783,202507,86794,0
9731746,6969058,202506,80585,0
6959721,5841558,202505,91940,0
15121657,7980258,202501,87588,1
2817563,2633792,202502,80369,0


In [10]:
item_features = (
    df_pos
    .groupby(['year_month', 'item_id'])
    .agg({
        'avg_price': 'mean',
        'brand': 'first',
        'category': 'first',
        'country': 'first',
        'inn': 'first',
        'owner': 'first'
    })
    .reset_index()
)

In [11]:
df_train_full = df_train.merge(
    item_features,
    on=['year_month', 'item_id'],
    how='left'
)

In [13]:
df_train_full = df_train_full.fillna({
    'avg_price': df_train_full['avg_price'].median(),
    'brand': 'unknown',
    'category': 'unknown',
    'country': 'unknown',
    'inn': 'unknown',
    'owner': 'unknown'
})

In [14]:
df_train_full.isna().sum()

user_id       0
year_month    0
item_id       0
label         0
avg_price     0
brand         0
category      0
country       0
inn           0
owner         0
dtype: int64

In [15]:
user_features = (
    df_pos
    .groupby(['user_id', 'year_month'])
    .agg({
        'cnt': 'sum',                      # сколько купил
        'item_id': 'nunique'               # разнообразие
    })
    .rename(columns={
        'cnt': 'user_total_cnt',
        'item_id': 'user_unique_items'
    })
    .reset_index()
)

In [16]:
df_train_full = df_train_full.merge(
    user_features,
    on=['user_id', 'year_month'],
    how='left'
)

In [18]:
print(df_train_full.sample(1, random_state=777).T.to_markdown())

|                   | 2906591                                  |
|:------------------|:-----------------------------------------|
| user_id           | 2662904                                  |
| year_month        | 202510                                   |
| item_id           | 2227                                     |
| label             | 0                                        |
| avg_price         | 371.2508620689655                        |
| brand             | Дюфалак                                  |
| category          | желудочно-кишечный тракт и обмен веществ |
| country           | США                                      |
| inn               | Лактулоза                                |
| owner             | Эбботт                                   |
| user_total_cnt    | 37.0                                     |
| user_unique_items | 26                                       |
